# Lecture 35 — Tidy Data
v.ekc

![Marie kondo](https://konmari.com/wp-content/uploads/2020/04/KonMari_Method101_MarieKondo_Hero_Landscape-scaled.jpg)

We are going to get your data in optimal form for visualization AKA `tidy`

By the end of class, you'll have a clean, tidy CSV ready to drop straight into Tableau.

**Today's sections:**
1. What is tidy data?
2. Load your project data
3. Diagnose: is it tidy?
4. Tidy your data
5. Export & hand off to Tableau

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

---
## 1. What Is Tidy Data?

Tidy data is a specific way of organizing a dataset that makes analysis, visualization, and sharing much easier. It was formalized by statistician Hadley Wickham, and it's the format that tools like Seaborn, plotnine, and Tableau all expect by default.



| Requirement | Example | What you DON't want |
|---|---|---|
| **Every column is a variable** | Each column represents exactly one thing being measured | Columns named `bp_2023`, `bp_2024`, `bp_2025`... |
| **Every row is an observation** | Each row represents one unit at one point in time/space | Multiple time points for the same unit crammed in one row |
| **Every cell is a single value** | Each cell contains exactly one data value | A cell like `"120/195"` or `"Male, 35"` |

### 👼 (Good) Example 1: Tidy Data

Here's a small health dataset. Each row is one patient in one year. Each column is one variable.

In [2]:
# Tidy data: one row per patient × year
tidy = pd.DataFrame({
    'patient':        ['Alice', 'Alice', 'Bob',  'Bob' ],
    'year':           [2023,    2024,    2023,    2024  ],
    'blood_pressure': [120,     118,     135,     130   ],
    'cholesterol':    [195,     188,     210,     205   ]
})
tidy

,patient,year,blood_pressure,cholesterol
0,Alice,2023,120,195
1,Alice,2024,118,188
2,Bob,2023,135,210
3,Bob,2024,130,205


**Every column is a variable**: `patient`, `year`, `blood_pressure`, `cholesterol` each represent exactly one thing.

**Every row is an observation**: each row is *one patient in one year* with the details for that year.

**Every cell is a single value**: no combined fields, no lists, no ratios.

This is exactly what Tableau, Seaborn, and plotnine expect when you say "x = year, y = blood_pressure, color = patient.

### 🤡 (BAD) Example 2: Column Headers Are Values (Wide Format)

Same data but in a wide shape. **Wide format**, and it's extremely common because it's how we humans naturally write tables in spreadsheets.

In [3]:
# Wide format: year is encoded in column names, not stored as data
wide = pd.DataFrame({
    'patient':   ['Alice', 'Bob'],
    'bp_2023':   [120,     135  ],
    'bp_2024':   [118,     130  ],
    'chol_2023': [195,     210  ],
    'chol_2024': [188,     205  ]
})
wide

,patient,bp_2023,bp_2024,chol_2023,chol_2024
0,Alice,120,118,195,188
1,Bob,135,130,210,205


**Column headers are values, not variable names** — `bp_2023`, `bp_2024`, `chol_2023`... the year is baked *into* the column name instead of stored as data.

> **Note:** If you can imagine the column name changing to a different value in the same category (e.g., `bp_2024` → `bp_2025`), it's probably a value masquerading as a variable name.

This format is hard to plot over time. Try telling Tableau "x = year" when year is spread across four different column names. 😬

**Fix:** `.melt()`.... we'll use this in Section 4.

### 💀 (BAD) Example 3: Multiple Variables Stored in One Column

What if someone tried to fix the wide format by melting it but stopped halfway? So a half-done tidying

In [4]:
# Multiple variables jammed into one column
half_tidy = pd.DataFrame({
    'patient': ['Alice',    'Alice',    'Alice',     'Alice',
                'Bob',      'Bob',      'Bob',       'Bob'   ],
    'metric':  ['bp_2023',  'bp_2024',  'chol_2023', 'chol_2024',
                'bp_2023',  'bp_2024',  'chol_2023', 'chol_2024'],
    'value':   [120,        118,        195,         188,
                135,        130,        210,         205         ]
})
half_tidy

,patient,metric,value
0,Alice,bp_2023,120
1,Alice,bp_2024,118
2,Alice,chol_2023,195
3,Alice,chol_2024,188
4,Bob,bp_2023,135
5,Bob,bp_2024,130
6,Bob,chol_2023,210
7,Bob,chol_2024,205


**(Bad) Multiple variables stored in one column**: `metric` is doing double duty: it encodes both the *measurement type* (`bp`, `chol`) and the *year* (`2023`, `2024`) in the same field.

> **How to tell:** If values in a column look like they could be split by `_` or `/` into two meaningful pieces, two variables are probably sharing a column.

**Fix:** `.str.split()` to separate into `measurement_type` and `year`, then `.pivot_table()` to spread the metrics back out.

### 💩 (Bad) Example 4: Multiple Values in One Cell

Sometimes data arrives with multiple measurements packed into a single cell — often a result of how it was exported.

In [5]:
# Two measurements packed into one cell
packed = pd.DataFrame({
    'patient': ['Alice',   'Bob'   ],
    'year':    [2023,      2023    ],
    'health_metrics': ['120/195', '135/210']  # blood_pressure / cholesterol
})
packed

,patient,year,health_metrics
0,Alice,2023,120/195
1,Bob,2023,135/210


**(Bad) Multiple values in one cell**: `health_metrics` contains both blood pressure *and* cholesterol, separated by `/`. You can't plot either variable without unpacking it first.

> **How to tell:** If a column contains separator characters like `/`, `;`, `,`, or `|` where you wouldn't expect them, two variables may be hiding inside each cell.

**Fix:** `.str.split(expand=True)` to unpack into separate columns.

---
### The 5 Most Common Problems with Messy Data

| # | Problem | Example | Fix |
|---|---|---|---|
| 1 | Column headers are values | `sales_jan`, `sales_feb`... | `.melt()` |
| 2 | Multiple variables in one column | `"bp_2023"` encodes two features: `blood pressure` and `year` | `.str.split()` + `.pivot_table()` |
| 3 | Variables stored in rows AND columns | Mix of long and wide | `.melt()` + `.pivot_table()` |
| 4 | Multiple values in one cell | `"120/195"` | `.str.split(expand=True)` |
| 5 | Related data split across multiple tables | Two datasets that belong together | `.merge()` |

Today we'll focus on problems 1–4. You've already practiced problem 5 earlier this semester.

---
## 2. Load Your Project Data

Time to bring in your own data. You've spent weeks cleaning and exploring your project dataset — now we're going to check if it's in tidy form and get it ready for Tableau.

### Step 0 — Export from your project notebook

**Open your project notebook in a separate tab.** Find your final cleaned dataframe (the one you've been using for your visualizations). Run this code at the bottom of that notebook, replacing `df_clean` with your actual dataframe name:

```python
# ── Paste this into your project notebook ──
# Replace 'df_clean' with the name of your cleaned dataframe
df_clean.to_csv('project_data.csv', index=False)
print("SUCCESS! My data is saved as project_data.csv")
```

Once that runs successfully, come back to this notebook. Your CSV should now be in the same JupyterHub folder. If not, move it to this working folder.

In [ ]:
# Load your project data — update the filename if yours is different
my_data = pd.read_csv('project_data.csv')
my_data.head()

### ETL — Quick Inspection

Before diagnosing, let's get oriented with the data.

In [ ]:
# Random sample — get a feel for the values
my_data.sample(5)

In [ ]:
# Data types and null counts
my_data.info()

In [ ]:
# Shape and summary statistics
print(f"Shape: {my_data.shape}")
my_data.describe()

---
## 3. Diagnose: Is Your Data Tidy?

Now let's evaluate your dataset against the three tidy principles. Don't worry if it isn't perfectly tidy — that's what Section 4 is for.

### 📋 Tidy Data Checklist

Before writing any code, read through these questions and think about your dataset:

| Question | If NO → potential problem |
|---|---|
| Are all column names variables (things measured), not values? | Column headers might be values → Problem #1 |
| Does each row represent exactly one observation? | Multiple units or time points per row → Problem #3 |
| Does each column represent exactly one variable? | Multiple variables packed together → Problem #2 |
| Is every cell a single, atomic value? | Packed values with separators → Problem #4 |

---
### Try It 1: Diagnose Your Data

1. Display the column names of your dataset. For each column: is this a **variable** (something measured), or a **value** (something that should be stored in a row)?

2. What does one row in your dataset represent? Write it out in plain English as a comment. Is every row the same kind of observation?

3. Scan the values in a few columns. Do any cells contain multiple values separated by `/`, `,`, `;`, or similar? Do any column names encode a value (like a year, month, or category)?

4. **Verdict:** Is your data tidy? If not, which problem(s) from the table above apply?

In [ ]:
# 1. Column names — variable or value?
print(my_data.columns.tolist())


In [ ]:
# 2. What does one row represent?
my_data.head(3)
# One row in my dataset represents: ...


In [ ]:
# 3. Check for packed values or value-encoded column names
# Look at a few columns closely — any separators? any column names that look like values?
my_data.head(3)


In [ ]:
# 4. Verdict (write your answer as a comment)
# My data is / is not tidy because:
# The main issue (if any) is Problem #:


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*This exercise is about your own data, so there's no single right answer. Here's what the diagnosis looks like using `gapminder_all.csv` as a reference.*

```python
# Example: diagnosing gapminder_all
gapminder = pd.read_csv('gapminder_all.csv')
print(gapminder.columns.tolist())
# → ['continent', 'country', 'gdpPercap_1952', 'gdpPercap_1957', ..., 'pop_2007']

# 1. Column names: 'gdpPercap_1952', 'gdpPercap_1957'...
#    These are VALUES (years + metrics), not variable names → Problem #1 and #2

# 2. One row = one country
#    But it should be one country × one year → Problem #3

# 3. No packed cells, but column headers encode years → Problem #1

# 4. Verdict: NOT tidy
#    Main issue: column headers are values (wide format) → Problem #1 + #2
```

</details>

---
## 4. Tidy Your Data

Now let's fix it. Below is a reference for the main reshaping tools, a full demo using gapminder, and then your turn.

### 📋 Board Reference — Reshaping Tools

| Tool | What it does | When to use it |
|---|---|---|
| `df.melt(id_vars=[...])` | Wide → long; column names become values in a new `variable` column | Column headers are values |
| `col.str.split('_', expand=True)` | Splits a string column into multiple columns | Multiple variables packed in one column |
| `df.pivot_table(index=[...], columns='var', values='val')` | Long → tidy; unique values in a column become column headers | Need to spread a variable back out |
| `pd.to_datetime(...)` | Converts strings to proper datetime type | Date stored as a string |

### Demo: Tidying Gapminder

Let's walk through the full pipeline on `gapminder_all.csv` — the same dataset from last lecture. You've seen the tidy version; now let's see how to get there from the raw wide format.

In [6]:
# Load the raw (messy) gapminder data
gapminder = pd.read_csv('gapminder_all.csv')
print(f"Shape: {gapminder.shape}")
gapminder.head(2)

Shape: (142, 38)


,continent,country,gdpPercap_1952,gdpPercap_1957,gdpPercap_1962,gdpPercap_1967,gdpPercap_1972,gdpPercap_1977,gdpPercap_1982,gdpPercap_1987,...,pop_1962,pop_1967,pop_1972,pop_1977,pop_1982,pop_1987,pop_1992,pop_1997,pop_2002,pop_2007
0,Africa,Algeria,2449.008185,3013.976023,2550.816880,3246.991771,4182.663766,4910.416756,5745.160213,5681.358539,...,11000948.0,12760499.0,14760787.0,17152804.0,20033753.0,23254956.0,26298373.0,29072015.0,31287142,33333216
1,Africa,Angola,3520.610273,3827.940465,4269.276742,5522.776375,5473.288005,3008.647355,2756.953672,2430.208311,...,4826015.0,5247469.0,5894858.0,6162675.0,7016384.0,7874230.0,8735988.0,9875024.0,10866106,12420476


In [7]:
# Step 1: melt() — wide → long
# id_vars = columns that are already proper variables (keep them as-is)
# everything else becomes a row
gapminder_long = gapminder.melt(id_vars=['continent', 'country'])
print(f"Shape after melt: {gapminder_long.shape}")
gapminder_long.head(8)

Shape after melt: (5112, 4)


,continent,country,variable,value
0,Africa,Algeria,gdpPercap_1952,2449.008185
1,Africa,Angola,gdpPercap_1952,3520.610273
2,Africa,Benin,gdpPercap_1952,1062.752200
3,Africa,Botswana,gdpPercap_1952,851.241141
4,Africa,Burkina Faso,gdpPercap_1952,543.255241
5,Africa,Burundi,gdpPercap_1952,339.296459
6,Africa,Cameroon,gdpPercap_1952,1172.667655
7,Africa,Central African Republic,gdpPercap_1952,1071.310713


In [8]:
# Step 2: str.split() — separate 'variable' into 'metric' and 'year'
# The column name 'gdpPercap_1952' encodes two things: metric + year
gapminder_long[['metric', 'year']] = gapminder_long['variable'].str.split('_', expand=True)
gapminder_long = gapminder_long.drop(columns='variable')
gapminder_long.head(8)

,continent,country,value,metric,year
0,Africa,Algeria,2449.008185,gdpPercap,1952
1,Africa,Angola,3520.610273,gdpPercap,1952
2,Africa,Benin,1062.752200,gdpPercap,1952
3,Africa,Botswana,851.241141,gdpPercap,1952
4,Africa,Burkina Faso,543.255241,gdpPercap,1952
5,Africa,Burundi,339.296459,gdpPercap,1952
6,Africa,Cameroon,1172.667655,gdpPercap,1952
7,Africa,Central African Republic,1071.310713,gdpPercap,1952


In [9]:
# Step 3: pivot_table() — spread 'metric' back into its own columns
gapminder_tidy = gapminder_long.pivot_table(
    index=['continent', 'country', 'year'],
    columns='metric',
    values='value'
).reset_index()

# Fix data types
gapminder_tidy['year'] = gapminder_tidy['year'].astype(int)
gapminder_tidy.columns.name = None  # clean up the pivot label

print(f"Final tidy shape: {gapminder_tidy.shape}")
gapminder_tidy.head(8)

Final tidy shape: (1704, 6)


,continent,country,year,gdpPercap,lifeExp,pop
0,Africa,Algeria,1952,2449.008185,43.077,9279525.0
1,Africa,Algeria,1957,3013.976023,45.685,10270856.0
2,Africa,Algeria,1962,2550.816880,48.303,11000948.0
3,Africa,Algeria,1967,3246.991771,51.407,12760499.0
4,Africa,Algeria,1972,4182.663766,54.518,14760787.0
5,Africa,Algeria,1977,4910.416756,58.014,17152804.0
6,Africa,Algeria,1982,5745.160213,61.368,20033753.0
7,Africa,Algeria,1987,5681.358539,65.799,23254956.0


NOW! Every column is a variable. Every row is one country $\times$ one year. Every cell is a single value.

Notice how the shape changed: `142 rows × 38 columns` → `1704 rows × 6 columns`. Tidy data is almost always *longer* and *narrower* than wide data.

---
### Try It 2: Tidy Your Data 😊

Apply the appropriate steps to your project data. **You may not need all of these steps** — use only what your diagnosis from Section 3 calls for.

1. **If column headers are values:** Use `.melt()`. Which columns are your `id_vars`?

2. **If multiple variables are packed in one column:** Use `.str.split()` to separate them into new columns.

3. **If you need to pivot variables back out into columns:** Use `.pivot_table()`.

4. Display your result and name it `my_tidy_data`. Verify: one column per variable, one row per observation, one value per cell.

In [ ]:
# 1. melt() .... if column headers are values
# my_tidy_data = my_data.melt(id_vars=['col1', 'col2'])


In [ ]:
# 2. str.split() .... if multiple variables are packed in one column
# my_tidy_data[['var1', 'var2']] = my_tidy_data['combined_col'].str.split('_', expand=True)
# my_tidy_data = my_tidy_data.drop(columns='combined_col')


In [ ]:
# 3. pivot_table() .... if you need to spread a variable back into columns
# my_tidy_data = my_tidy_data.pivot_table(
#     index=['id_col1', 'id_col2', 'var2'],
#     columns='var1', values='value'
# ).reset_index()


In [ ]:
# 4. Display your tidy result
# If your data was already tidy, just run: my_tidy_data = my_data.copy()
my_tidy_data = ...
my_tidy_data.head()

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Everyone's project data is different, so there's no single right answer. The gapminder pipeline above is your best reference for what each step produces. Here's the general pattern:*

```python
# If your data has wide format (column headers are values):
my_tidy_data = my_data.melt(id_vars=['id_col1', 'id_col2'])

# If a column packs multiple variables:
my_tidy_data[['var1', 'var2']] = my_tidy_data['combined'].str.split('_', expand=True)
my_tidy_data = my_tidy_data.drop(columns='combined')

# If you need to pivot:
my_tidy_data = my_tidy_data.pivot_table(
    index=['id_col1', 'id_col2', 'var2'],
    columns='var1', values='value'
).reset_index()
my_tidy_data.columns.name = None

# If your data was already tidy:
my_tidy_data = my_data.copy()
```

*Many right approaches — the goal is to satisfy the three tidy principles.*

</details>

---
## 5. Export & now off to Tableau~~~!

Your tidy data is ready. Let's save it as a CSV and download it. Then we'll move to Tableau.

In [11]:
from IPython.display import FileLink, display

# Save your tidy data as a CSV
my_tidy_data.to_csv('project_data_tidy.csv', index=False)

# Click the link below to download it to your computer
display(FileLink('project_data_tidy.csv'))
print(f"Saved! Shape: {my_tidy_data.shape}")
print("Click the link above to download your tidy CSV :) we'll import this into Tableau next.")

/Users/emilychang/Documents/CPH teaching/data271_sp26/lectures/project_data_tidy.csv

Saved! Shape: (1704, 6)
Click the link above to download your tidy CSV :) we'll import this into Tableau next.


---
## Appendix — Tidy Data Quick Reference

### Minimal template — diagnose
```python
df.head()           # Look at overall structure
df.columns.tolist() # Are column names variables or values?
df.info()           # Data types and null counts
```

### Full tidying pipeline
```python
import pandas as pd

# 1. Wide → long
df_long = df.melt(id_vars=['id_col1', 'id_col2'])

# 2. Split a packed column (e.g., 'metric_2023' → 'metric', '2023')
df_long[['var1', 'var2']] = df_long['variable'].str.split('_', expand=True)
df_long = df_long.drop(columns='variable')

# 3. Long → tidy (spread var1 back into columns)
df_tidy = df_long.pivot_table(
    index=['id_col1', 'id_col2', 'var2'],
    columns='var1',
    values='value'
).reset_index()
df_tidy.columns.name = None

# 4. Fix data types if needed
df_tidy['year'] = df_tidy['year'].astype(int)

# 5. Export
from IPython.display import FileLink, display
df_tidy.to_csv('my_tidy_data.csv', index=False)
display(FileLink('my_tidy_data.csv'))
```

📚 Further reading: [Tidy Data — Hadley Wickham (2014)](https://vita.had.co.nz/papers/tidy-data.pdf)